> Rubrics as Rewards

- RLVR (RL with Verifiable Rewards) 在数学和代码领域取得了巨大成功，因为这些领域有明确的正确/错误信号（Verify correctness）。
    - But... 在医学诊断、科学推理等真实世界任务中，答案往往没有单一的“真理”。评价一个回答好坏，往往依赖于多个维度的细微判断（Nuanced, multi-criteria judgments）。
- RaR：“好答案的标准”解构为一系列可解释的、细粒度的检查清单（Checklist）。

### RaR

对于输入 prompt $x$ ，模型生成回答 $\hat{y} \sim \pi_{\theta}(\cdot|x)$ 。RaR 为每个 $x$ 关联一组评分细则 $\{(w_j, c_j)\}_{j=1}^k$ ，其中：

* $w_j \in \mathbb{R}$ ：第 $j$ 条标准的权重 (Weight)。
* $c_j : (x, \hat{y}) \mapsto \{0, 1\}$ ：二元正确性函数，判断是否满足该标准。

Explicit Aggregation (显式聚合)：直接计算加权满足率，标准化到 `[0,1]`。

$$
r(x, \hat{y}) = \frac{\sum_{j=1}^k w_j \cdot c_j(x, \hat{y})}{\sum_{j=1}^k w_j}
$$

Implicit Aggregation (隐式聚合): 将所有 Rubric criteria 作为上下文输入给 LLM Judge，由模型直接打分。

$$
r_{implicit}(x, \hat{y}) = f_{\phi}(x, \hat{y}, \{d_j\}_{j=1}^k)
$$

### RaR vs. LLM-as-judge(Direct-Likert baseline)

- 通用 vs. 实例特异 (Generic vs. Instance-Specific)
    - Standard LLM-as-judge: 通常使用一套通用的 System Prompt（例如：“请给这个回答打分，1-10分，标准是准确性、有用性...”）。它依赖模型内在的通用知识来判断好坏，非常主观。
    - RaR: 对于每一个具体的训练 Prompt，都会先生成一套专属的评分细则 (Rubrics)（例如针对“治疗酸中毒”的问题，细则会明确要求：“1. 是否计算了碳酸氢盐量？2. 是否提到了分次给药以避免副作用？”）。Judge 必须根据这些具体条目来打分。
- 黑盒 vs. 结构化 (Black-box vs. Structured)
    - Standard LLM-as-judge: 输出通常是一个标量分数 (Scalar Score)。模型为什么给 7 分而不是 8 分？往往是不可解释的“黑盒”，容易受回答长度、语气等肤浅特征影响（Reward Hacking）。
    - RaR: 强迫 Judge 进行结构化分解。它将“好坏”拆解为多个二元（Binary）或细粒度的检查点。
        - 这种方式将一个主观的推理任务（Subjective Reasoning），转化为了近似于数学题的伪客观验证任务（Pseudo-Verifiable Task），这也是论文标题 "Beyond Verifiable Domains" 的深意——在没有标准答案的领域，人造出标准。
- 对 Judge 能力的要求 (Capability Requirements)
    - Standard LLM-as-judge: 高度依赖 Judge 模型本身的推理上限。如果 Judge 比较弱（比如 7B 模型），它很难直接分辨出复杂的逻辑漏洞，打分会非常随机。
    - RaR: 极大地降低了 Judge 的门槛。论文实验（Figure 3）表明，一旦有了清晰的 Rubrics 作为“脚手架”，即使是 7B 的小模型也能做出接近 GPT-4 级别的准确评审。这使得用小模型监督大模型训练（Weak-to-Strong Generalization）成为可能。

### system prompt

- rubrics generation prompt
    - Grounded in Expert Guidance.
    - Comprehensive Coverage.
    - Criterion Importance.
    - Self-Contained Evaluation.
- rating prompt
    - Prompt for RAR-IMPLICIT Method

### 模型选择

> strong to weak: Weak Judge + Rubrics $\approx$ Strong Judge。

- rubrics generation：gpt-4o
- Policy model：qwen2.5-7b-instruct/llama-3.1-8b-instruct
- judge/reward model: 同等规模甚至更小的模型（qwen2.5-7b/3b）配合 rubrics 进行打分；